# Zadanie 1 
**Polecenie**: Oblicz różnicę xG (expected goals) dla każdego stanu meczu (wygrana-przegrana-remis) dla obu drużyn.

In [7]:
import pandas as pd

In [13]:
df=pd.read_csv('Polonia Bytom_Pogo  Grodzisk Mazowiecki_4068759.csv', sep=',')
df_clean=df.drop_duplicates(subset='id').sort_values(by=['period','timestamp','index'])
df_przefiltrowane = df_clean.query("event_type_id==16")
print(df_przefiltrowane.shape)

(34, 248)


In [9]:
df_wybrane=df_przefiltrowane[['team_id','statsbomb_xg','outcome_id']]
print(df_wybrane)

      team_id  statsbomb_xg  outcome_id
132     35037      0.058596        98.0
164     35037      0.100382        96.0
524     35037      0.028285        98.0
679     35037      0.042174       100.0
782     35036      0.197502        97.0
953     35036      0.054080        96.0
958     35036      0.063239        96.0
1069    35037      0.043911       100.0
1073    35037      0.035480        97.0
1103    35036      0.040728        96.0
1139    35037      0.035418        98.0
1261    35037      0.062603        98.0
1384    35036      0.005402       100.0
1409    35037      0.092842        97.0
1487    35037      0.066237        98.0
1518    35036      0.006998       100.0
1705    35036      0.009596        97.0
1821    35037      0.007875        96.0
2017    35037      0.022412        96.0
2100    35037      0.032257        96.0
2111    35037      0.029686        98.0
2334    35036      0.034193        98.0
2464    35037      0.056432        96.0
2470    35037      0.078858        98.0


In [ ]:
#97 gol
#brak informacji o id goli samobójczych
#35036 pogon 35037 polonia 
polonia=0
pogon=0
xg_pogon={"Wygrana":0.0,"Remis":0.0,"Przegrana":0.0}
xg_polonia={"Wygrana":0.0,"Remis":0.0,"Przegrana":0.0}
for i,klub,xg,gol in df_wybrane.itertuples(index=True):
    if pogon==polonia:
        stan_pogon,stan_polonia="Remis","Remis"
    elif pogon>polonia:
        stan_pogon,stan_polonia="Wygrana","Przegrana"
    else:
        stan_pogon,stan_polonia="Przegrana","Wygrana"
    
    if klub==35036:
        xg_pogon[stan_pogon]+=xg
        if gol==97:
            pogon+=1
    else:
        xg_polonia[stan_polonia]+=xg
        if gol==97:
            polonia+=1

wynik_pogon=[
    round(xg_pogon["Wygrana"]-xg_polonia["Przegrana"],3),
    round(xg_pogon["Remis"]-xg_polonia["Remis"],3),
    round(xg_pogon["Przegrana"]-xg_polonia["Wygrana"],3)
]
wynik_polonia=[
    round(xg_polonia["Wygrana"]-xg_pogon["Przegrana"],3),
    round(xg_polonia["Remis"]-xg_pogon["Remis"],3),
    round(xg_polonia["Przegrana"]-xg_pogon["Wygrana"],3)
]

wynik=pd.DataFrame(
    [wynik_pogon,wynik_polonia],
    columns=["Wygrana","Remis","Przegrana"],
    index=["Pogoń Grodzisk Mazowiecki","Polonia Bytom"]
)
print(wynik)

                           Wygrana  Remis  Przegrana
Pogoń Grodzisk Mazowiecki    0.038 -0.314     -0.050
Polonia Bytom                0.050  0.314     -0.038


## Podsumowanie
Po obliczeniu różnic w xG drużyn obserwujemy, że oba zespoły po strzeleniu bramki dającej prowadzenie wypracowywały minimalnie lepsze sytuacje niż przegrywający przeciwnik. Różnice wyniosły odpowiednio +0.038 dla Pogoń Grodzisk Mazowiecki i +0.050 dla Polonia Bytom. Pokazuje to, że drużyny po osiągnięciu korzystnego rezultatu nie zostawały spychane do obrony (wedłu xG) tylko tworzyły równie dobre sytuacje co przeciwnik. Analogicznie w momentach gdy zespoły przegrywały, miały gorszy bilans xG niż drużyna przeciwna. Podczas remisu, to drużyna Polonii Bytom tworzyła lepsze sytuacje niż Pogoń Grodzisk Mazowiecki, gdyż różnica oczekiwanych bramek wynosi 0.314.

# Zadanie 2
**Polecenie**: Dostajesz pytanie, czy zawodnik X nadaje się na wahadłowego w naszym systemie. Masz dane StatsBomb z jego 20 meczów. Opisz, jak byś do tego podszedł i gdzie te dane Cię zawiodą.

Zakładając że zawodnik X w tych 20 meczach zagrał wystarczającą ilość minut do rzetelnego zobrazowania jego umiejętności oraz stylu gry, pierwszym krokiem jest sprawdzenie statystyk naszych obecnych wahadłowych, aby porównać ich statystyki ze statystykami zawodnika X. Znaczna część statystyk przeliczana będzie na 90 minut (oznaczone *), co zniweluje różnice w rozegranych minutach. Nastepnym krokiem będzie sprawdzenie statystyk StatsBomb:

## Faza z piłką
* **Podania** - Obliczenie celności podań zestawionych ze wskaźnikiem trudności wykonywanych podań. Sprawdzenie liczby przerzutów ciężaru gry na drugą stronę*.

* **Efektywność zdobywania przestrzeni** - Sprawdzenie liczby i długości prowadzeń piłki* oraz wyliczenie odsetka udanych dryblingów.

* **Tworzenie sytuacji strzeleckich** - Porównanie liczby asytst* do wskaźnika oczekiwanych asyst*. Obliczenie odsetka celnych dośrodkowań oraz sprawdzenie stref z których zawodnik X dośrodkowuje.

* **Decyzje** - Analiza wskaźnika wyceny decyzji na boisku, sprawdzenie jak bardzo zachowania gracza z piłką powodują wzrost szans na zdobycie gola.

* **Efektywność pod presją** - Ocena jakości decyzji gracza pod presją.

* **Straty** - Analiza strat piłki* z rozróżnieniem stref oraz sprawdzenie jak to wpłynęło na szanse zdobycia gola przez przeciwnika.

## Faza bez piłki
* **Pressing** - Sprawdzenie liczby kontrpressingów* oraz pressingów* z rozróżnieniem stref.

* **Odbiory** - Wyliczenie odsetka wygranych pojedynków w obronie oraz sprawdzenie liczby odbiorów*, odzyskanych piłek* oraz przecięć podań*.

* **Ustawienie** - Analiza modelu odpowiedzialności defensywnej do oceny doskoku oraz ustawienia się w obronie.

* **Kary indywidualne** - Sprawdzenie liczby kartek* (żółtych i czerwonych) otrzymanych przez zawodnika X.

* **Błędy w obronie** - Sprawdzenie w ilu sytuacjach* gracz miał największe prawdopodobieństwo zawinienia straconej bramki.

## Ograniczenia danych StatsBomb
* **Motoryka** - Brak informacji o motoryce gracza: ile wynosi jego maksymalny sprint, ile przebiegł kilometrów i z jaką intensywnością. Te dane są kluczowe w celu zweryfikowania czy zawodnik X nadaje się do naszego systemu gry, ponieważ gracz może mieć bardzo dobre statystyki ale kondycyjnie nie wytrzymywać 90 minut.

* **Postawa** - Brak informacji o postawie gracza podczas różnych momentów meczu: jak reaguje na własne błędy, czy jest liderem zespołu, jak reaguje na niekorzystny wynik meczu.

* **Ruch bez piłki** - Nie mamy danych, które pokazują np. wybiegnięcia na obieg w których gracz nie otrzymał piłki czy trzymania linii spalonego.

* **Styl gry zespołu** - Zespół gracza X może grać innym systemem niż nasz, co powoduje pewne zniekształcenia jego statystyk. Jeżeli gracz występował jako typowy lewy obrońca jego statystyki ofensywne mogą być gorsze, ze względu na niższe ustawienie w formacji.

* **Kontuzje** - Brak informacji o historii kontuzji.

## Podsumowanie
Dane ze StatsBomb dają dobry fundament pod selekcję graczy pasujących do naszego systemu gry, jednak z kilkoma lukami. Powinny być uzupełnione o informacje motoryczne, które pokażą jak zawodnik wygląda pod względem biegowym. Ważnym uzupełnieniem informacji o graczu będą również wideo klipy, które pokażą ustawianie się oraz postawę niewerbalną w meczu.